In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import sys
from sklearn.preprocessing import StandardScaler

# Ensure the required libraries are accessible for model loading
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

RANDOM_STATE = 42
print("Libraries loaded.")

In [ ]:
# --- Configuration ---
# The actual test data file that will be provided for submission
TEST_DATA_PATH = "test.csv"
OUTPUT_FILE = "submission.csv"
LIMIT_PCT = 0.20 # The maximum allowed percentage of predicted bankrupt companies

# --- Model Paths (Using the names confirmed in the repository) ---
# 1. Preprocessing (Task A)
PREPROCESS_JOBLIB = "preprocess_for_clustering.joblib"
FEATURES_JOBLIB = "top_features_for_clustering.joblib"

# 2. Cluster-ID Model (Task 3.3.1, built by B)
# Note: Using the actual filename confirmed in the repository
CLUSTER_ID_MODEL = "cluster_4_model.joblib"

# 3. Subgroup Stacking Models (Task 3.3.2)
MODEL_A_PATH = "cluster0_stacking_PNH.joblib" # Cluster 0 (A's model)
MODEL_B_PATH = "cluster4_stacking_B.joblib"    # Cluster 4 (B's model)
MODEL_C_PATH = "cluster2_stacking_C.joblib"    # Cluster 2 (C's model)

print("Configuration set.")

In [ ]:
def load_pipeline_from_joblib(path, key=None):
    """Robustly loads a joblib object (pipeline, model, or package dict)."""
    if not os.path.exists(path):
        print(f"FATAL ERROR: Model file {path} not found.")
        sys.exit(1) # Exit if critical files are missing

    obj = joblib.load(path)

    if isinstance(obj, dict) and key:
        return obj.get(key)

    return obj

print("--- Loading All Resources ---")

# 1. Load Scaler & Feature Names (Task 3.1)
preprocess_pkg = load_pipeline_from_joblib(PREPROCESS_JOBLIB)
scaler = preprocess_pkg.get("scaler") if isinstance(preprocess_pkg, dict) else preprocess_pkg
feature_names = load_pipeline_from_joblib(FEATURES_JOBLIB)

# 2. Load Cluster-ID Model (Task 3.3.1)
clf_cluster = load_pipeline_from_joblib(CLUSTER_ID_MODEL, key="model")

# 3. Load Subgroup Models (Task 3.3.2)
model_0 = load_pipeline_from_joblib(MODEL_A_PATH, key="pipeline")
model_2 = load_pipeline_from_joblib(MODEL_C_PATH, key="pipeline")
model_4 = load_pipeline_from_joblib(MODEL_B_PATH, key="pipeline")

print("✅ All models and preprocessing components loaded successfully.")

In [ ]:
# --- Load Test Data (Crucial Step) ---
if not os.path.exists(TEST_DATA_PATH):
    print(f"FATAL ERROR: Test data file '{TEST_DATA_PATH}' not found.")
    print("Please ensure the test file is in the same directory before submission.")
    sys.exit(1)

df_test = pd.read_csv(TEST_DATA_PATH)
print(f"Test data loaded. Shape: {df_test.shape}")

# Ensure 'Index' is the first column for safety (though it should be)
if 'Index' not in df_test.columns:
    print("FATAL ERROR: 'Index' column not found in test data.")
    sys.exit(1)

# --- Feature Selection & Scaling (Task 3.1 Preprocessing) ---
try:
    X_test_raw = df_test[feature_names].copy()
    X_test_scaled = scaler.transform(X_test_raw)
    print("Test data scaled successfully using A's fitted scaler.")
except KeyError as e:
    print(f"FATAL ERROR: Test CSV missing required feature: {e}")
    sys.exit(1)

In [ ]:
# 1. Predict Cluster ID (Task 3.3.1)
test_clusters = clf_cluster.predict(X_test_scaled)
df_test['predicted_cluster'] = test_clusters

print("\nCluster Assignments on Test Data:")
print(df_test['predicted_cluster'].value_counts().sort_index())

# 2. Initialize Prediction Storage
test_probs = np.zeros(len(df_test))

# 3. Routing Logic (Task 3.3.2)
print("\n--- Routing Test Rows to Subgroup Models ---")

for cluster_id in sorted(df_test['predicted_cluster'].unique()):
    mask = df_test['predicted_cluster'] == cluster_id
    X_subset = df_test.loc[mask, feature_names] # Pass unscaled features to Pipeline
    count = mask.sum()

    if count == 0: continue

    print(f"Processing Cluster {cluster_id}: {count} rows")

    if cluster_id == 0 and model_0:
        # Model A: Cluster 0
        test_probs[mask] = model_0.predict_proba(X_subset)[:, 1]
    elif cluster_id == 4 and model_4:
        # Model B: Cluster 4
        test_probs[mask] = model_4.predict_proba(X_subset)[:, 1]
    elif cluster_id == 2 and model_2:
        # Model C/D: Cluster 2
        test_probs[mask] = model_2.predict_proba(X_subset)[:, 1]
    elif cluster_id in [1, 5]:
        # Constant 0: Clusters 1 (0 bankrupt) and 5 (0 bankrupt)
        print("  -> Constant 0 (Safe Cluster)")
        test_probs[mask] = 0.0
    elif cluster_id in [3, 6]:
        # Constant 1: Clusters 3 (1 bankrupt) and 6 (3 bankrupt) - High-risk tiny clusters
        print("  -> Constant 1 (Risky Cluster)")
        test_probs[mask] = 1.0 # Highest probability
    else:
        print(f"  ⚠️ Unknown Cluster ID {cluster_id}. Defaulting to 0.0 probability.")
        test_probs[mask] = 0.0

df_test['pred_prob'] = test_probs

In [ ]:
# --- Post-Processing: Enforce < 20% Bankrupt Constraint ---

# 1. Initial Hard Predictions (Standard threshold 0.5)
final_preds = (test_probs > 0.5).astype(int)
num_positive = final_preds.sum()
pct_positive = num_positive / len(df_test)

print(f"\nInitial Predictions: {num_positive} bankrupts ({pct_positive:.2%})")

# 2. Check Constraint
if pct_positive > LIMIT_PCT:
    print(f"⚠️ VIOLATION: Predictions exceed {LIMIT_PCT:.0%}!")

    # Strategy: Keep only the top K most confident predictions
    # Calculate the maximum allowed count (using 19% to be safe)
    target_count = int(len(df_test) * (LIMIT_PCT - 0.01))

    # Sort indices by prediction probability (descending)
    sorted_indices = np.argsort(test_probs)[::-1]

    # Create new predictions array, defaulted to 0
    new_preds = np.zeros(len(df_test), dtype=int)

    # Set top K indices to 1 (Bankrupt)
    top_k_indices = sorted_indices[:target_count]
    new_preds[top_k_indices] = 1

    final_preds = new_preds
    print(f"✅ Adjusted Predictions: {final_preds.sum()} bankrupts ({final_preds.sum()/len(df_test):.2%})")
else:
    print("✅ Constraint Met.")

df_test['Bankrupt?'] = final_preds

# --- Export Submission File ---
submission = df_test[['Index', 'Bankrupt?']].copy()

# Final Sanity Checks
assert submission.shape[0] == df_test.shape[0]
assert submission['Bankrupt?'].isnull().sum() == 0
assert submission['Bankrupt?'].mean() < LIMIT_PCT

submission.to_csv(OUTPUT_FILE, index=False)
print("-" * 30)
print(f"Submission saved to {OUTPUT_FILE} in required format (Index,Bankrupt?).")
print(f"Final Count: {submission['Bankrupt?'].sum()} bankruptcies predicted.")
print("-" * 30)